In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Parity of SNAP parameter 

In [2]:
tEQM,tEMM,tNatom = np.loadtxt("validation/train_parity_energy.txt",unpack=True)
tFQM,tFMM = np.loadtxt("validation/train_parity_force.txt",unpack=True)
vEQM,vEMM,vNatom = np.loadtxt("validation/valid_parity_energy.txt",unpack=True)
vFQM,vFMM = np.loadtxt("validation/valid_parity_force.txt" ,unpack=True)
normtEQM = np.true_divide(tEQM,tNatom)
normtEMM = np.true_divide(tEMM,tNatom)
normvEQM = np.true_divide(vEQM,vNatom)
normvEMM = np.true_divide(vEMM,vNatom)
fig, (epair,fpair) = plt.subplots(1,2,figsize=(8, 4))
fig.tight_layout(w_pad=5)
epair.set_title('Energy Parity')
boundx = [-10.5, -4.7]
boundy = [-10.5, -4.7]
epair.plot(boundx, boundy, color='black',linewidth=1)
epair.scatter(normtEQM, normtEMM, marker='o', color='royalblue', s=3, label='Training')
epair.scatter(normvEQM, normvEMM, marker='o', color='darkorange', s=3, label='Validation')
epair.set_xlim(boundx)
epair.set_ylim(boundy)
epair.set(xlabel='DFT Energy/atom (eV)',ylabel='SNAP Energy/atom (eV)')
epair.legend(loc="upper left",fontsize=11)
################################

for line in open("validation/Record_train.txt").readlines():
    parts = line.split()
    if parts[0] == 'Energy': T_ERMSE = parts[3]
    elif parts[0] == 'Force':T_FRMSE = parts[3]
    elif parts[0] == 'Probability': T_AP, protocol = parts[8].split('%')
for line in open("validation/Record_valid.txt").readlines():
    parts = line.split()
    if parts[0] == 'Energy': V_ERMSE = parts[3]
    elif parts[0] == 'Force':V_FRMSE = parts[3]
    elif parts[0] == 'Probability': V_AP, protocol = parts[8].split('%')
################################    
str_TE = 'RMSE = %.3f meV/atom' % (float(T_ERMSE)*1000)
str_VE = 'RMSE = %.3f meV/atom' % (float(V_ERMSE)*1000)
str_TF = 'RMSE = %.3f eV/Å' % float(T_FRMSE)
str_VF = 'RMSE = %.3f eV/Å' % float(V_FRMSE)
str_TA = '%% of Force pair Ang. < 20° = %.3f %%' % float(T_AP)
str_VA = '%% of Force pair Ang. < 20° = %.3f %%' % float(V_AP)
epair.text(0.30, 0.12,str_TE,color='royalblue',fontsize=11,horizontalalignment='left',verticalalignment='center', transform=epair.transAxes)
epair.text(0.30, 0.04,str_VE,color='darkorange',fontsize=11,horizontalalignment='left',verticalalignment='center', transform=epair.transAxes)
boundx = [-25, 25]
boundy = [-25, 25]
fpair.set_title('Force Parity')
fpair.plot(boundx, boundy, color='black',linewidth=1)
fpair.scatter(tFQM, tFMM, marker='o', color='royalblue', s=3, label='Training')
fpair.scatter(vFQM, vFMM, marker='o', color='darkorange', s=3, label='Validation')
fpair.set_xlim(boundx)
fpair.set_ylim(boundy)
fpair.set(xlabel='DFT Force (eV/A)',ylabel='SNAP Force (eV/A)')
fpair.legend(loc="upper left",fontsize=11)
fpair.text(0.45, 0.12,str_TF,color='royalblue',fontsize=11,horizontalalignment='left',verticalalignment='center', transform=fpair.transAxes)
fpair.text(0.45, 0.04,str_VF,color='darkorange',fontsize=11,horizontalalignment='left',verticalalignment='center', transform=fpair.transAxes)
fpair.text(1.10, 0.12,str_TA,color='royalblue',fontsize=11,horizontalalignment='left',verticalalignment='center', transform=fpair.transAxes)
fpair.text(1.10, 0.04,str_VA,color='darkorange',fontsize=11,horizontalalignment='left',verticalalignment='center', transform=fpair.transAxes)

plt.show()

# RDF between DFT and SNAP

In [3]:
r1, rdf_dft = np.loadtxt("RDF_AIMD.txt",skiprows=2,unpack=True)
r2, rdf_snap = np.loadtxt("RDF_SNAP.txt",skiprows=2,unpack=True)
ax = plt.subplot(1,1,1)
ax.set_title('Radial Distribution Function of HEA in 300K')
ax.plot(r1, rdf_dft, color='black',linewidth=1,label='AIMD')
ax.plot(r2, rdf_snap, color='red',linewidth=1,label='SNAP MD')
ax.set(xlabel='r (Å)',ylabel='g (r)')
plt.show()

# SS Curve from 300K to 900K

In [4]:
nsys = ["Random <100>","Random <111>"]
fpos = ["random_100","random_111"]
for n in range(len(nsys)):
    strain1, sx1, sy1, sz1 = np.loadtxt("elastic_properties/"+str(fpos[n])+"_300K/sample1/SS_curve.txt",skiprows=1,unpack=True)
    strain2, sx2, sy2, sz2 = np.loadtxt("elastic_properties/"+str(fpos[n])+"_600K/sample1/SS_curve.txt",skiprows=1,unpack=True)
    strain3, sx3, sy3, sz3 = np.loadtxt("elastic_properties/"+str(fpos[n])+"_900K/sample1/SS_curve.txt",skiprows=1,unpack=True)    
    ax = plt.subplot(1,1,1)
    ax.set_title('Stress Strain Curve of '+str(nsys[n]))
    ax.plot(strain1, sx1, color='blue',linewidth=1,label='300K')
    ax.plot(strain2, sx2, color='green',linewidth=1,label='600K')
    ax.plot(strain3, sx3, color='red',linewidth=1,label='900K')    
    ax.set_xlabel('Strain')
    ax.set_ylabel('Stress(GPa)')
    plt.legend(loc='center right',fontsize=11)
    slope1 = np.polyfit(strain1[:21],sx1[:21],1)[0]
    slope2 = np.polyfit(strain2[:21],sx2[:21],1)[0]
    slope3 = np.polyfit(strain3[:21],sx3[:21],1)[0]    
    string1 = 'E_300K = %.3f GPa' % slope1
    string2 = 'E_600K = %.3f GPa' % slope2
    string3 = 'E_900K = %.3f GPa' % slope3    
    ax.text(0.02, 0.78, string1,color='blue',fontsize=11,horizontalalignment='left',verticalalignment='center',transform=ax.transAxes)
    ax.text(0.02, 0.72, string2,color='green',fontsize=11,horizontalalignment='left',verticalalignment='center',transform=ax.transAxes)
    ax.text(0.02, 0.66, string3,color='red',fontsize=11,horizontalalignment='left',verticalalignment='center',transform=ax.transAxes)    
    plt.show()



# Thermal expa

In [5]:
from statistics import mean, stdev
temp = [300, 600, 900]
lx_300K = [46.900227,46.905154,46.903157]
ly_300K = [46.903477,46.896129,46.900366]
lz_300K = [50.032815,50.035966,50.034178]
lx_600K = [47.030333,47.032667,47.031149]
ly_600K = [47.031029,47.025122,47.027281]
lz_600K = [50.167742,50.171592,50.169896]
lx_900K = [47.165282,47.166673,47.164885]
ly_900K = [47.162375,47.162213,47.16194]
lz_900K = [50.314134,50.312908,50.312045]
L_300K = [a/15. for a in lx_300K]+ [a/15. for a in ly_300K] + [a/16. for a in lz_300K]
L_600K = [a/15. for a in lx_600K]+ [a/15. for a in ly_600K] + [a/16. for a in lz_600K]
L_900K = [a/15. for a in lx_900K]+ [a/15. for a in ly_900K] + [a/16. for a in lz_900K]
avg_L = [mean(L_300K),mean(L_600K),mean(L_900K)]
avg_dL = [100*(avg_L[0]-avg_L[0])/avg_L[0], 100*(avg_L[1]-avg_L[0])/avg_L[0], 100*(avg_L[2]-avg_L[0])/avg_L[0]]
dev_L = [stdev(L_300K),stdev(L_600K),stdev(L_900K)]
fig, ax = plt.subplots()
ax.set_xlabel('Temperature (K)')
#ax.set_ylabel('Lattice constane (Å)')
ax.set_ylabel('dL/L$_{0}$ (%)')
ax.scatter(temp, avg_dL, color='red')
#ax.errorbar(temp, avg_L, yerr=dev_L,fmt="o",color='red')
ax.set_xlim([200,1000])
ax.set_ylim([-0.05,1.0])
slope = np.polyfit(temp,avg_dL,1)[0]
res = np.polyfit(temp,avg_dL,1)[1]
curve_x = np.arange(250,1000,50)
curve_y = slope*curve_x+res
ax.plot(curve_x,curve_y,linestyle='dashed',color='red',linewidth=1)
print(slope)
string1 = str(r'$\alpha$$_{exp.}$ = ')+'11.4 x 10$^{-6}$ K$^{-1}$'
string2 = str(r'$\alpha$$_{DFT}$ = ')+'8.1 x 10$^{-6}$ K$^{-1}$'
string3 = str(r'$\alpha$$_{SNAP}$ = ')+'9.3 x 10$^{-6}$ K$^{-1}$'
ax.text(0.05, 0.8, string1,color='black',fontsize=14,horizontalalignment='left',verticalalignment='center',transform=ax.transAxes)
ax.text(0.05, 0.7, string2,color='black',fontsize=14,horizontalalignment='left',verticalalignment='center',transform=ax.transAxes)
ax.text(0.05, 0.6, string3,color='red',fontsize=14,horizontalalignment='left',verticalalignment='center',transform=ax.transAxes)
plt.show()

0.0009312801781716675


# Monte Carlo Sampling

In [6]:
nsys = ["a:<100> b:<100> c:<100>","a:<100> b:<110> c:<110>","a:<111> b:<112> c:<110>"]
fpos = ["mc_x100_y100_z100","mc_x100_y110_z110","mc_x111_y112_z110"]
natoms = [7200,7920,7776]
for n in range(len(nsys)):
    plt.title('MC Sampling for '+str(nsys[n]))
    step1, pe1 = np.loadtxt("standard_model/"+str(fpos[n])+"/MC_300K/mc_folder/MC_record.txt",usecols=(0,1),unpack=True)
    step2, pe2 = np.loadtxt("standard_model/"+str(fpos[n])+"/MC_900K/mc_folder/MC_record.txt",usecols=(0,1),unpack=True)
    step3, pe3 = np.loadtxt("standard_model/"+str(fpos[n])+"/MC_1200K/mc_folder/MC_record.txt",usecols=(0,1),unpack=True)
    plt.plot(step1, pe1/natoms[n], color='blue',linewidth=1,label="MC_300K")
    plt.plot(step2, pe2/natoms[n], color='darkorange',linewidth=1,label="MC_900K")
    plt.plot(step3, pe3/natoms[n], color='red',linewidth=1,label="MC_1200K")
    #plt.xlim(-10000, 500000)
    #plt.ylim(-152000, -150500)
    plt.xlabel('MC step (x10$^{6}$)')
    plt.ylabel('SNAP Energy (eV/atom)')
    plt.legend(loc='best',fontsize=11)
    plt.show()

# Warran-Cowley Parameter

In [7]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns

nsys = ["Random","MC 1200K","MC 900K", "MC 300K"]
"""
path = ["standard_model/random_x100_y100_z100/sample1/MD_300K/WC.csv",\
        "standard_model/mc_x100_y100_z100/MC_1200K/MD_300K/WC.csv",\
        "standard_model/mc_x100_y100_z100/MC_900K/MD_300K/WC.csv",\
        "standard_model/mc_x100_y100_z100/MC_300K/MD_300K/WC.csv"
       ]
"""
path = ["standard_model/random_x111_y112_z110/sample1/MD_300K/WC.csv",\
        "standard_model/mc_x111_y112_z110/MC_1200K/MD_300K/WC.csv",\
        "standard_model/mc_x111_y112_z110/MC_900K/MD_300K/WC.csv",\
        "standard_model/mc_x111_y112_z110/MC_300K/MD_300K/WC.csv"
       ]
for n in range(len(nsys)):
    file_path = path[n]
    data = pd.read_csv(file_path, index_col=0)
    # Extract the relevant 5x5 matrix for the heatmap
    heatmap_data = data.loc[['Ni', 'Co', 'Ti', 'Zr', 'Hf'], ['Ni', 'Co', 'Ti', 'Zr', 'Hf']]
    #heatmap_data = heatmap_data.astype(float)
    #print(heatmap_data)
    # Create the heatmap and save it
    plt.figure(figsize=(6, 4))
    ax = sns.heatmap(heatmap_data, annot=True,annot_kws={"size": 14}, fmt=".3f", cmap="coolwarm", center=0, vmin=-0.5, vmax=0.5)
    ax.set(xlabel="Central Atom", ylabel="Neighboring Atom")
    plt.title(nsys[n])
    plt.tight_layout()
    plt.show()

SS Curve from 300K to 900K

In [8]:
nsys = ["MC_300K <100>","MC_900K <100>","MC_1200K <100>","MC_300K <111>","MC_900K <111>","MC_1200K <111>"]
fpos = ["MC300K_100","MC900K_100","MC1200K_100","MC300K_111","MC900K_111","MC1200K_111"]
for n in range(len(nsys)):
    strain1, sx1, sy1, sz1 = np.loadtxt("elastic_properties/"+str(fpos[n])+"_300K/SS_curve.txt",skiprows=1,unpack=True)
    strain2, sx2, sy2, sz2 = np.loadtxt("elastic_properties/"+str(fpos[n])+"_600K/SS_curve.txt",skiprows=1,unpack=True)
    strain3, sx3, sy3, sz3 = np.loadtxt("elastic_properties/"+str(fpos[n])+"_900K/SS_curve.txt",skiprows=1,unpack=True)    
    ax = plt.subplot(1,1,1)
    ax.set_title('Stress Strain Curve of '+str(nsys[n]))
    ax.plot(strain1, sx1, color='blue',linewidth=0.5,label='300K')
    ax.plot(strain2, sx2, color='green',linewidth=0.5,label='600K')
    ax.plot(strain3, sx3, color='red',linewidth=0.5,label='900K')    
    ax.set_xlabel('Strain')
    ax.set_ylabel('Stress(GPa)')
    plt.legend(loc='center right',fontsize=11)
    slope1 = np.polyfit(strain1[:21],sx1[:21],1)[0]
    slope2 = np.polyfit(strain2[:21],sx2[:21],1)[0]
    slope3 = np.polyfit(strain3[:21],sx3[:21],1)[0]    
    string1 = 'E_300K = %.3f GPa' % slope1
    string2 = 'E_600K = %.3f GPa' % slope2
    string3 = 'E_900K = %.3f GPa' % slope3    
    ax.text(0.02, 0.78, string1,color='blue',fontsize=11,horizontalalignment='left',verticalalignment='center',transform=ax.transAxes)
    ax.text(0.02, 0.72, string2,color='green',fontsize=11,horizontalalignment='left',verticalalignment='center',transform=ax.transAxes)
    ax.text(0.02, 0.66, string3,color='red',fontsize=11,horizontalalignment='left',verticalalignment='center',transform=ax.transAxes)    
    plt.show()


In [9]:
from statistics import mean, stdev
nsys = ["Random <100>","Random <111>"]
fpos = ["random_100","random_111"]
title = ["<100>","<111>"]
#MC300_100 = [118.769, 116.999, 112.361]
#MC900_100 = [122.426, 115.994, 107.943]
#MC1200_100 = [116.554, 109.519, 100.314]
#MC300_111 = [97.647, 97.882, 90.021]
#MC900_111 = [105.443, 104.913, 93.415]
#MC1200_111 = [103.835, 103.916, 92.682]
MC300_100 = [121.611, 117.156, 115.929]
MC900_100 = [117.257, 118.182, 116.579]
MC1200_100 = [121.347, 110.846, 113.209]
MC300_111 = [104.589, 104.803, 96.200]
MC900_111 = [105.970, 105.924, 97.080]
MC1200_111 = [106.614, 106.465, 96.655]
MC300 = [MC300_100, MC300_111]
MC900 = [MC900_100, MC900_111]
MC1200 = [MC1200_100, MC1200_111]

for n in range(len(nsys)):
    elas300 = []
    elas600 = []
    elas900 = []
    for i in range(5):
        strain1, sx1, sy1, sz1 = np.loadtxt("elastic_properties/"+str(fpos[n])+"_300K/sample"+str(i+1)+"/SS_curve.txt",skiprows=1,unpack=True)
        strain2, sx2, sy2, sz2 = np.loadtxt("elastic_properties/"+str(fpos[n])+"_600K/sample"+str(i+1)+"/SS_curve.txt",skiprows=1,unpack=True)
        strain3, sx3, sy3, sz3 = np.loadtxt("elastic_properties/"+str(fpos[n])+"_900K/sample"+str(i+1)+"/SS_curve.txt",skiprows=1,unpack=True)    
        slope1 = np.polyfit(strain1[:21],sx1[:21],1)[0]
        slope2 = np.polyfit(strain2[:21],sx2[:21],1)[0]
        slope3 = np.polyfit(strain3[:21],sx3[:21],1)[0]
        elas300.append(slope1)
        elas600.append(slope2)
        elas900.append(slope3)
    print(elas300)
    print(elas600)
    print(elas900)
    avg_elas = [mean(elas300),mean(elas600),mean(elas900)]
    dev_elas = [stdev(elas300),stdev(elas600),stdev(elas900)]
    fig, ax = plt.subplots()
    ax.set_title("Young's Modulus in "+str(title[n])+" direction")
    temp = [300, 600, 900]
    ax.plot(temp, avg_elas, color='black',marker='o',markersize=5,linewidth=1,label="Random")
    ax.errorbar(temp, avg_elas, yerr=dev_elas,ecolor='black',color='black',elinewidth=2,capsize=4)
    ax.plot(temp, MC1200[n], '-o', color='red',markersize=5,linewidth=1,label='MC 1200K')
    ax.plot(temp, MC900[n], '-o', color='green',markersize=5,linewidth=1,label='MC 900K')
    ax.plot(temp, MC300[n], '-o', color='blue',markersize=5,linewidth=1,label='MC 300K')
    ax.set_xlabel('Temperaure (K)')
    ax.set_ylabel('Young\'s Modulus (GPa)')
    plt.legend(loc='upper right',fontsize=11)
    ###############
    
    plt.show()

[114.18577374244596, 114.4873540296435, 114.12732192102524, 114.2315177325934, 114.34102560915044]
[115.49535173217079, 115.88210445617631, 115.61520333440026, 115.80823438740495, 115.83712141625965]
[114.81550690868234, 114.92337825682809, 114.90564773614709, 114.90659585717435, 114.68965146533601]


[91.69116735222946, 91.52375451847486, 92.01185512461012, 92.4901715347594, 91.96329922779628]
[92.27433876251764, 92.2342198737125, 92.64003333913082, 92.67841864408263, 92.61670599934324]
[88.08423690286601, 88.15068924082688, 88.22319039949375, 88.58103992100807, 88.28326843206673]


# Determine CRSS from NEB calculation

In [ ]:
import subprocess
# nsys = ["Edge_{010}<100> 300K","Edge_{110}<100> 300K","Edge_{110}<111> 300K"]
# fpos = ["edge_b100_p100_300K","edge_b100_p110_300K","edge_b111_p110_300K"]
# direct = [1,-1,1]
# n_curves = [24,11,30]
# #n_curves = [12,12,12]
# disloc_length = [46.9034771478331,48.6360184445399,45.9438543483382]
# fpath = "./"
# 
# for sys in range(len(nsys)):
#     s_stress = []
#     s_strain = []
#     curr_strain = 0
#     E_b = []
#     fig, (neb_plot,crss_pred,ss) = plt.subplots(1,3,figsize=(12, 4))
#     fig.tight_layout(w_pad=5)    
#     neb_plot.set_title('NEB Curves of '+str(nsys[sys]))    
#     cmap = plt.get_cmap('rainbow')    
#     for i in range(n_curves[sys]):
#         E_file = str(fpath)+"/"+str(fpos[sys])+"/sample1/CRSS_strain/step_"+str(i)+"/NEB_energy.txt"
#         with open(E_file, 'r') as file:
#             E_neb = [float(line.strip()) for line in file.readlines()]
#         E_relate = [ (E-E_neb[0])/disloc_length[sys] for E in E_neb ]
#         E_b.append(max(E_relate)*1000)
#         fstdout = str(fpath)+'/'+str(fpos[sys])+'/sample1/CRSS_strain/step_'+str(i)+'/STDOUT1'
#         p = subprocess.Popen(['grep', 'Curr_INIT',fstdout], stdout=subprocess.PIPE)
#         p_line = p.communicate()[0].decode().split()
#         curr_strain += abs(float(p_line[1]))
#         s_strain.append(curr_strain)
#         s_stress.append(direct[sys]*float(p_line[2])*1000)
#         color = cmap(i/n_curves[sys])
#         coord = range(10)
#         neb_plot.plot(coord, E_relate, color=color,linewidth=1,label='strain_'+str(i))
#         neb_plot.set(xlabel='Coordinate',ylabel='Normalized Relative Energy (eV/A)')
#         neb_plot.legend(loc='lower right',fontsize=6)
#     boundy = [-10, max(E_b)+10]
#     crss_pred.set_ylim(boundy)
#     crss_pred.scatter(s_stress, E_b, marker='o', color='royalblue', s=10)
#     crss_pred.set(xlabel='Shear Stress (MPa)',ylabel='Energy Barrier (meV/A)')
#     
#     ss.scatter(s_strain, s_stress, marker='o', color='darkorange', s=10)
#     ss.set(xlabel='Shear Strain',ylabel='Shear Stress (MPa)')

# Only plot Edge_{010}<100> 300K
nsys = ["Edge_{110}<100> 300K","Edge_{111}<100> 300K"]
fpos = ["edge_b100_p110_300K","edge_b111_p110_300K"]
direct = [-1,1]
n_curves = [18,15]
disloc_length = [48.640287349669,45.9438543483382]
sample_num = [1,1]
fpath = "."

sys = 0
s_stress = []
s_strain = []
curr_strain = 0
E_b = []
fig, (neb_plot,crss_pred,ss) = plt.subplots(1,3,figsize=(12, 4))
fig.tight_layout(w_pad=5)
neb_plot.set_title('NEB Curves of '+str(nsys[sys]))
cmap = plt.get_cmap('rainbow')
for i in range(n_curves[sys]):
    if i == 0:
        continue
    E_file = str(fpath)+"/"+str(fpos[sys])+"/sample"+str(sample_num[sys])+"/CRSS_strain/step_"+str(i)+"/NEB_energy.txt"
    try:
        with open(E_file, 'r') as file:
            # Handle both formats: one float per line OR multiple space-separated floats per line
            E_neb = []
            for line in file.readlines():
                line = line.strip()
                if not line:  # Skip empty lines
                    continue
                # Split by whitespace and convert each value to float
                values = line.split()
                E_neb.extend([float(val) for val in values])
        print(f"{i}: {len(E_neb)}")
        if len(E_neb) == 0:
            print(f"Warning: Empty file: {E_file}, skipping step {i}")
            continue
    except FileNotFoundError:
        print(f"Warning: File not found: {E_file}, skipping step {i}")
        continue
    E_relate = [ (E-E_neb[0])/disloc_length[sys] for E in E_neb ]
    E_b.append(max(E_relate)*1000)
    fstdout = str(fpath)+'/'+str(fpos[sys])+'/sample2/CRSS_strain/step_'+str(i)+'/STDOUT1'
    p = subprocess.Popen(['grep', 'Curr_INIT',fstdout], stdout=subprocess.PIPE)
    p_line = p.communicate()[0].decode().split()
    curr_strain += abs(float(p_line[1]))
    s_strain.append(curr_strain)
    s_stress.append(direct[sys]*float(p_line[2])*1000)
    color = cmap(i/n_curves[sys])
    coord = range(10)
    neb_plot.plot(coord, E_relate, color=color,linewidth=1,label='strain_'+str(i))
    neb_plot.set(xlabel='Coordinate',ylabel='Normalized Relative Energy (eV/A)')
    neb_plot.legend(loc='lower right',fontsize=6)
boundy = [-10, max(E_b)+10]
crss_pred.set_ylim(boundy)
crss_pred.scatter(s_stress, E_b, marker='o', color='royalblue', s=10)
crss_pred.set(xlabel='Shear Stress (MPa)',ylabel='Energy Barrier (meV/A)')

ss.scatter(s_strain, s_stress, marker='o', color='darkorange', s=10)
ss.set(xlabel='Shear Strain',ylabel='Shear Stress (MPa)')

FileNotFoundError: [Errno 2] No such file or directory: './edge_b100_p110_300K/sample1/CRSS_strain/step_1/NEB_energy.txt'

In [ ]:
import subprocess
nsys = ["Screw_{010}<100> 300K"]
fpos = ["screw_b100_p100_300K"]
n_curves = [30]
sample_num = [1]
disloc_length = [46.8984507685887]
direct = [1]
fpath = "."

for sys in range(len(nsys)):
    s_stress = []
    s_strain = []
    curr_strain = 0
    E_b = []
    fig, (neb_plot,crss_pred,ss) = plt.subplots(1,3,figsize=(12, 4))
    fig.tight_layout(w_pad=5)    
    neb_plot.set_title('NEB Curves of '+str(nsys[sys]))    
    cmap = plt.get_cmap('rainbow')    
    for i in range(n_curves[sys]):
        if i == 0:
            continue
        E_file = str(fpath)+"/"+str(fpos[sys])+"/sample"+str(sample_num[sys])+"/CRSS_strain/step_"+str(i)+"/NEB_energy.txt"
        try:
            with open(E_file, 'r') as file:
                # Handle both formats: one float per line OR multiple space-separated floats per line
                E_neb = []
                for line in file.readlines():
                    line = line.strip()
                    if not line:  # Skip empty lines
                        continue
                    # Split by whitespace and convert each value to float
                    values = line.split()
                    E_neb.extend([float(val) for val in values])
            if len(E_neb) == 0:
                print(f"Warning: Empty file: {E_file}, skipping step {i}")
                continue
            E_relate = [ (E-E_neb[0])/disloc_length[sys] for E in E_neb ]
        except FileNotFoundError:
            print(f"Warning: File not found: {E_file}, skipping step {i}")
            continue
        
        E_b.append(max(E_relate)*1000)
        fstdout = str(fpath)+'/'+str(fpos[sys])+'/sample'+str(sample_num[sys])+'/CRSS_strain/step_'+str(i)+'/STDOUT1'
        p = subprocess.Popen(['grep', 'Curr_INIT',fstdout], stdout=subprocess.PIPE)
        p_line = p.communicate()[0].decode().split()
        curr_strain += abs(float(p_line[1]))
        s_strain.append(curr_strain)
        s_stress.append(direct[sys]*float(p_line[2])*1000)
        color = cmap(i/n_curves[sys])
        coord = range(10)
        neb_plot.plot(coord, E_relate, color=color,linewidth=1,label='strain_'+str(i))
        neb_plot.set(xlabel='Coordinate',ylabel='Normalized Relative Energy (eV/A)')
        neb_plot.legend(loc='lower right',fontsize=6)
    boundy = [-10, max(E_b)+10]
    crss_pred.set_ylim(boundy)
    crss_pred.scatter(s_stress, E_b, marker='o', color='royalblue', s=10)
    crss_pred.set(xlabel='Shear Stress (MPa)',ylabel='Energy Barrier (meV/A)')
    
    ss.scatter(s_strain, s_stress, marker='o', color='darkorange', s=10)
    ss.set(xlabel='Shear Strain',ylabel='Shear Stress (MPa)')
        

ValueError: x and y must have same first dimension, but have shapes (10,) and (7,)

# Dislocation Dynamics (MD+MIN)

In [12]:
wkdir = "dislocation/DD_md"
nsys = ["Edge_{010}<100> 300K","Edge_{110}<100> 300K","Edge_{110}<111> 300K"]
fpos = ["edge_b100_p100_300K","edge_b100_p110_300K","edge_b111_p110_300K"]
fig, ax = plt.subplots(1,len(nsys),figsize=(15, 5))
fig.tight_layout(w_pad=5)
for n in range(len(nsys)):
    txt = str(wkdir)+"/"+str(fpos[n])
    strain, sxz = np.loadtxt(str(txt)+"/SS_curve.txt",usecols=(0,5),skiprows=1,unpack=True) 
    ax[n].plot(strain, sxz, color='black',marker='o',markersize=5,linewidth=1)
    ax[n].set_title("SS Curve of "+str(nsys[n]))
    ax[n].set_xlabel('Strain')
    ax[n].set_ylabel('Shear Stress (GPa)') 
    #ax[n].set_xlim(0.0, 0.2)
    #ax[n].set_ylim(-0.5, 3.0)
    #ax[n].legend(loc='upper right',fontsize=11)
plt.show()


# Dislocation Dynamics (MD+tfMC+MIN)

In [13]:
wkdir = "dislocation/DD_tfmc"
nsys = ["Edge_{010}<100>","Edge_{110}<100>","Edge_{110}<111>"]
fpos = ["edge_b100_p100","edge_b100_p110","edge_b111_p110"]

fig, ax = plt.subplots(1,len(nsys),figsize=(15, 5))
fig.tight_layout(w_pad=5)    
for n in range(len(nsys)):
    txt = str(wkdir)+"/"+str(fpos[n])
    strain1, sxz1 = np.loadtxt(str(txt)+"_300K/SS_curve.txt",usecols=(0,5),skiprows=1,unpack=True) 
    strain2, sxz2 = np.loadtxt(str(txt)+"_900K/SS_curve.txt",usecols=(0,5),skiprows=1,unpack=True) 
    ax[n].plot(strain1, sxz1, color='black',marker='o',markersize=5,linewidth=1)
    ax[n].plot(strain2, sxz2, color='red',marker='o',markersize=5,linewidth=1)
    ax[n].set_title("SS Curve of "+str(nsys[n]))
    ax[n].set_xlabel('Strain')
    ax[n].set_ylabel('Shear Stress (GPa)') 
    #ax[n].set_xlim(0.0, 0.2)
    #ax[n].set_ylim(-0.5, 3.0)
    #ax[n].legend(loc='upper right',fontsize=11)
plt.show()

wkdir = "dislocation/DD_tfmc"
nsys = ["Screw_{010}<100>","Screw_{110}<100>","Screw_{110}<111>"]
fpos = ["screw_b100_p100","screw_b100_p110","screw_b111_p110"]

fig, ax = plt.subplots(1,len(nsys),figsize=(15, 5))
fig.tight_layout(w_pad=5)    
for n in range(len(nsys)):
    txt = str(wkdir)+"/"+str(fpos[n])
    strain1, sxz1 = np.loadtxt(str(txt)+"_300K/SS_curve.txt",usecols=(0,5),skiprows=1,unpack=True) 
    strain2, sxz2 = np.loadtxt(str(txt)+"_900K/SS_curve.txt",usecols=(0,5),skiprows=1,unpack=True) 
    ax[n].plot(strain1, sxz1, color='black',marker='o',markersize=5,linewidth=1)
    ax[n].plot(strain2, sxz2, color='red',marker='o',markersize=5,linewidth=1)
    ax[n].set_title("SS Curve of "+str(nsys[n]))
    ax[n].set_xlabel('Strain')
    ax[n].set_ylabel('Shear Stress (GPa)') 
    #ax[n].set_xlim(0.0, 0.2)
    #ax[n].set_ylim(-0.5, 3.0)
    #ax[n].legend(loc='upper right',fontsize=11)
plt.show()

# Dislocation Dynamics (tfMC+MIN)

In [14]:
wkdir = "dislocation/DD_tfmc2"
nsys = ["Edge_{010}<100> 300K","Edge_{110}<100> 300K","Edge_{110}<111> 300K"]
fpos = ["edge_b100_p100_300K","edge_b100_p110_300K","edge_b111_p110_300K"]
fig, ax = plt.subplots(1,len(nsys),figsize=(15, 5))
fig.tight_layout(w_pad=5)
for n in range(len(nsys)):
    txt = str(wkdir)+"/"+str(fpos[n])
    strain, sxz = np.loadtxt(str(txt)+"/SS_curve.txt",usecols=(0,5),skiprows=1,unpack=True) 
    ax[n].plot(strain, sxz, color='black',marker='o',markersize=5,linewidth=1)
    ax[n].set_title("SS Curve of "+str(nsys[n]))
    ax[n].set_xlabel('Strain')
    ax[n].set_ylabel('Shear Stress (GPa)') 
    #ax[n].set_xlim(0.0, 0.2)
    #ax[n].set_ylim(-0.5, 3.0)
    #ax[n].legend(loc='upper right',fontsize=11)
plt.show()